<a href="https://colab.research.google.com/github/Minh-Khuong/Bai-tap-nhom---GPA/blob/main/GPA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit pandas numpy scikit-learn matplotlib seaborn
!npm install localtunnel

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import os

st.set_page_config(page_title="Hệ Thống Dự Đoán AI GPA V1.0", page_icon="⚡", layout="wide")

st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@500;700;900&family=Chakra+Petch:wght@500;600;700&display=swap');

    /* Nền ma trận Cyberpunk chiều sâu vô cực */
    .stApp {
        background-color: #030508;
        background-image:
            linear-gradient(rgba(0, 243, 255, 0.03) 1px, transparent 1px),
            linear-gradient(90deg, rgba(0, 243, 255, 0.03) 1px, transparent 1px);
        background-size: 40px 40px;
        font-family: 'Chakra Petch', sans-serif;
        color: #c3dafe;
    }

    /* Thiết kế tiêu đề HUD phát sáng cao cấp, không dùng ký tự rác */
    h1, h2, h3, h4 {
        font-family: 'Orbitron', sans-serif !important;
        text-transform: uppercase;
        color: #00f3ff !important;
        text-shadow: 0 0 8px rgba(0, 243, 255, 0.6), 0 0 20px rgba(0, 119, 255, 0.4);
        letter-spacing: 2px;
    }

    /* Hộp Glassmorphism Bo Mạch Điện Tử */
    div[data-testid="stVerticalBlock"] > div {
        background: rgba(10, 17, 32, 0.85) !important;
        border: 1px solid rgba(0, 243, 255, 0.3);
        border-radius: 12px;
        padding: 25px;
        box-shadow: 0 0 25px rgba(0, 243, 255, 0.1);
        backdrop-filter: blur(15px);
    }

    /* Thanh điều hướng Tabs phong cách thiết bị tối tân */
    .stTabs [data-baseweb="tab-list"] {
        gap: 10px;
        background-color: #090d16;
        padding: 8px;
        border-radius: 8px;
        border: 1px solid rgba(0, 243, 255, 0.1);
    }
    .stTabs [data-baseweb="tab"] {
        font-family: 'Orbitron', sans-serif !important;
        color: #64748b !important;
        background-color: transparent !important;
        font-weight: 700;
        padding: 10px 20px;
        transition: all 0.3s ease;
    }
    .stTabs [data-baseweb="tab"][aria-selected="true"] {
        color: #ff007f !important;
        border-bottom: 2px solid #ff007f !important;
        text-shadow: 0 0 8px rgba(255, 0, 127, 0.5);
    }

    /* Thiết lập thanh trượt tinh chỉnh */
    .stSlider > div > div > div > div { background-color: #ff007f !important; }
    .stSlider > div > div > div > div > div { background-color: #00f3ff !important; box-shadow: 0 0 12px #00f3ff !important;}

    /* Nút bấm Động Cơ Hồi Quy Tuyến Tính siêu ngầu */
    div.stButton > button:first-child {
        background: linear-gradient(90deg, #ff007f 0%, #7928ca 50%, #00f3ff 100%);
        background-size: 200% auto;
        color: #ffffff !important;
        border: 1px solid rgba(0, 243, 255, 0.4) !important;
        font-family: 'Orbitron', sans-serif !important;
        font-weight: 900;
        font-size: 22px;
        padding: 1.3rem;
        width: 100%;
        border-radius: 6px;
        text-transform: uppercase;
        letter-spacing: 4px;
        transition: 0.4s cubic-bezier(0.25, 0.8, 0.25, 1);
        box-shadow: 0 0 20px rgba(255, 0, 127, 0.4);
    }
    div.stButton > button:first-child:hover {
        background-position: right center;
        box-shadow: 0 0 35px rgba(0, 243, 255, 0.8), 0 0 50px rgba(255, 0, 127, 0.5);
        transform: translateY(-2px);
    }

    /* Khung Hiển Thị Hologram Kết Quả */
    .cyber-result-box {
        background: rgba(2, 4, 8, 0.95);
        border: 1px solid;
        border-radius: 10px;
        padding: 40px 20px;
        text-align: center;
        margin-top: 30px;
    }
    .gpa-score {
        font-family: 'Orbitron', sans-serif;
        font-size: 110px;
        font-weight: 900;
        margin: 10px 0;
        text-shadow: 0 0 20px currentColor, 0 0 45px currentColor;
    }
    .gpa-insight {
        font-family: 'Chakra Petch', sans-serif;
        font-size: 20px;
        font-weight: 700;
        background: rgba(0,0,0,0.7);
        padding: 14px 30px;
        border-radius: 6px;
        display: inline-block;
        color: #ffffff;
        border-left: 5px solid currentColor;
        border-right: 5px solid currentColor;
        letter-spacing: 0.5px;
    }
    hr { border-color: rgba(0, 243, 255, 0.2); }
</style>
""", unsafe_allow_html=True)
@st.cache_data
def load_data():
    file_name = "GPA_Survey_Data.csv"
    if os.path.exists(file_name):
        try:
            df = pd.read_csv(file_name)
            if len(df.columns) == 11:
                df.columns = ['Timestamp', 'Email', 'So_Mon_Hoc', 'Gio_Hoc', 'Lam_Them', 'Gio_Ngu', 'Tham_Gia_CLB', 'Ty_Le_Di_Hoc', 'Cach_Hoc', 'Gio_MXH', 'GPA']
                df = df.drop(columns=['Timestamp', 'Email'], errors='ignore')
                df['Lam_Them'] = df['Lam_Them'].astype(str).map({'Có': 1, 'Không': 0}).fillna(0)
                df['Tham_Gia_CLB'] = df['Tham_Gia_CLB'].astype(str).map({'Có': 1, 'Không': 0}).fillna(0)
                df['Hoc_Nhom'] = df['Cach_Hoc'].astype(str).map({'Học nhóm': 1, 'Tự học': 0}).fillna(0)
                df.drop(columns=['Cach_Hoc'], inplace=True)
                def cvt_pct(val):
                    if isinstance(val, str) and '%' in val: return float(val.replace('%', '')) / 100.0
                    return float(val)
                df['Ty_Le_Di_Hoc'] = df['Ty_Le_Di_Hoc'].apply(cvt_pct)
            elif len(df.columns) == 9:
                for col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
            return df.dropna(), "DATA: CONNECTED SECURELY"
        except Exception:
            pass
    np.random.seed(42)
    n_samples = 300
    gio_hoc = np.clip(np.random.normal(4.5, 2, n_samples), 0, 12)
    ty_le_di_hoc = np.random.choice([0.0, 0.25, 0.5, 0.75, 1.0], n_samples, p=[0.05, 0.1, 0.15, 0.35, 0.35])
    gio_mxh = np.clip(np.random.normal(4.5, 2.2, n_samples), 0, 12)
    gio_ngu = np.clip(np.random.normal(7, 1.3, n_samples), 4, 11)

    gpa = 2.4 + (gio_hoc * 0.08) + (ty_le_di_hoc * 0.65) - (gio_mxh * 0.03)
    gpa += np.random.normal(0, 0.15, n_samples)

    df = pd.DataFrame({
        'So_Mon_Hoc': np.random.randint(4, 9, n_samples),
        'Gio_Hoc': gio_hoc,
        'Lam_Them': np.random.randint(0, 2, n_samples),
        'Gio_Ngu': gio_ngu,
        'Tham_Gia_CLB': np.random.randint(0, 2, n_samples),
        'Ty_Le_Di_Hoc': ty_le_di_hoc,
        'Gio_MXH': gio_mxh,
        'Hoc_Nhom': np.random.randint(0, 2, n_samples),
        'GPA': np.clip(gpa, 2.0, 4.0).round(2)
    })
    return df, "DATA: EMULATED OVERRIDE"

@st.cache_resource
def train_model(df):
    X = df.drop(columns=['GPA'])
    y = df['GPA']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LinearRegression()
    model.fit(X_train, y_train)
    return model, X, r2_score(y_test, model.predict(X_test)), np.sqrt(mean_squared_error(y_test, model.predict(X_test)))

df, status_msg = load_data()
model, X_data, r2_score_val, rmse_val = train_model(df)
st.markdown("<h1 style='text-align: center; font-size: 3.5rem; margin-bottom: 0;'>AI GPA PREDICTOR</h1>", unsafe_allow_html=True)
st.markdown(f"<p style='text-align: center; color: #ff007f; font-size: 15px; font-weight: 700; letter-spacing: 4px;'>{status_msg}</p>", unsafe_allow_html=True)
st.markdown("---")

tab1, tab2 = st.tabs(["AI PREDICTION", "MODEL ANALYTICS"])

with tab1:
    col1, col2 = st.columns(2, gap="large")
    with col1:
        st.markdown("<h3 style='color: #00f3ff; font-size: 22px; margin-bottom: 20px;'>Hồ Sơ Học Tập</h3>", unsafe_allow_html=True)
        so_mon = st.slider("Số lượng môn học đang đăng ký học kỳ này:", min_value=1, max_value=12, value=5, step=1)
        di_hoc_opts = {'Vắng học hoàn toàn (0%)': 0.0, 'Nghỉ học nhiều (25%)': 0.25, 'Đi học trung bình (50%)': 0.5, 'Chăm chỉ đi học (75%)': 0.75, 'Đi học tuyệt đối (100%)': 1.0}
        ty_le_di_hoc = di_hoc_opts[st.selectbox("Tỷ lệ điểm danh tham gia trên lớp:", options=list(di_hoc_opts.keys()), index=4)]
        gio_hoc = st.slider("Số giờ tự ôn tập nghiên cứu kiến thức / Ngày:", min_value=0.0, max_value=15.0, value=4.0, step=0.5)
        cach_hoc = 1 if st.radio("Phương pháp tiếp cận bài học chính:", options=['Hình thức Tự học cá nhân độc lập', 'Hình thức Tham gia tương tác nhóm'], index=0, horizontal=True) == 'Hình thức Tham gia tương tác nhóm' else 0

    with col2:
        st.markdown("<h3 style='color: #ff007f; font-size: 22px; margin-bottom: 20px;'>Thói Quen Sinh Hoạt</h3>", unsafe_allow_html=True)
        gio_ngu = st.slider("Thời gian ngủ trung bình mỗi ngày:", min_value=2.0, max_value=14.0, value=7.0, step=0.5)
        gio_mxh = st.slider("Thời gian giải trí và mạng xã hội / Ngày:", min_value=0.0, max_value=15.0, value=3.0, step=0.5)
        lam_them = 1 if st.radio("Tình trạng đi làm thêm kiếm thu nhập:", options=['Không đi làm thêm', 'Có công việc làm thêm bên ngoài'], index=0, horizontal=True) == 'Có công việc làm thêm bên ngoài' else 0
        clb = 1 if st.radio("Hoạt động phong trào Đoàn hội, Câu lạc bộ:", options=['Không tham gia', 'Có tham gia hoạt động năng nổ'], index=0, horizontal=True) == 'Có tham gia hoạt động năng nổ' else 0

    st.markdown("<br>", unsafe_allow_html=True)
    predict_btn = st.button("Kích Hoạt Thuật Toán Phân Tích")

    if predict_btn:
        input_data = pd.DataFrame([{
            'So_Mon_Hoc': so_mon, 'Gio_Hoc': gio_hoc, 'Lam_Them': lam_them,
            'Gio_Ngu': gio_ngu, 'Tham_Gia_CLB': clb, 'Ty_Le_Di_Hoc': ty_le_di_hoc,
            'Gio_MXH': gio_mxh, 'Hoc_Nhom': cach_hoc
        }])
        pred_gpa = max(0.0, min(4.0, model.predict(input_data)[0]))

        if pred_gpa >= 3.6: msg, color = "Mô hình phân tích: Thói quen tối ưu đạt mức xuất sắc vượt trội.", "#00ffcc"
        elif pred_gpa >= 3.2: msg, color = "Mô hình phân tích: Bạn đang kiểm soát trạng thái cân bằng rất tốt.", "#b824ff"
        elif pred_gpa >= 2.5: msg, color = "Mô hình phân tích: Kết quả trung bình khá, cần siết chặt lại kỷ luật.", "#ff9900"
        else: msg, color = "Mô hình phân tích: Các thông số lối sống có rủi ro nghiêm trọng cho học tập.", "#ff007f"

        progress_pct = int((pred_gpa / 4.0) * 100)

        st.markdown(f"""
        <div class="cyber-result-box" style="border-color: {color}; color: {color}; box-shadow: 0 0 35px {color}44 inset;">
            <div style="font-family: 'Orbitron', sans-serif; font-size: 22px; font-weight: 700; letter-spacing: 3px; color: {color}; margin-bottom: 15px;">
                PREDICTED GPA OUTPUT
            </div>
            <div style="width: 70%; background-color: #111827; border-radius: 10px; margin: 0 auto 25px auto; border: 1px solid {color}55; padding: 2px;">
                <div style="width: {progress_pct}%; background-color: {color}; height: 14px; border-radius: 10px; box-shadow: 0 0 15px {color};"></div>
            </div>
            <div class="gpa-score">{pred_gpa:.2f}</div>
            <div class="gpa-insight" style="color: {color}; border-color: {color};">{msg}</div>
        </div>
        """, unsafe_allow_html=True)
with tab2:
    st.markdown(f"#### Kiểm định độ chuẩn xác lõi thuật toán")
    st.markdown(f"""
    <div style='background: rgba(10, 17, 32, 0.85); padding: 20px; border-radius: 10px; border: 1px solid rgba(0, 243, 255, 0.3); margin-bottom: 25px;'>
        <p style='margin: 0; font-size: 16px;'>Hệ số xác định R-squared (Độ tin cậy): <span style='color:#00f3ff; font-weight:bold;'>{r2_score_val:.4f}</span></p>
        <p style='margin: 8px 0 0 0; font-size: 16px;'>Sai số toàn phương trung bình (RMSE): <span style='color:#ff0055; font-weight:bold;'>{rmse_val:.4f}</span></p>
    </div>
    """, unsafe_allow_html=True)

    col_chart1, col_chart2 = st.columns(2)
    plt.style.use("dark_background")

    with col_chart1:
        st.markdown("<h4 style='text-align: center; color: #94a3b8; margin-bottom: 10px;'>Ma Trận Hệ Số Tương Quan Thực Tế</h4>", unsafe_allow_html=True)
        fig1, ax1 = plt.subplots(figsize=(8, 6), facecolor='#0a1120')
        ax1.set_facecolor('#0a1120')
        corr = df.corr()
        cmap = sns.color_palette("blend:#00f3ff,#0a1120,#ff007f", as_cmap=True)
        sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)), annot=True, cmap=cmap, fmt=".2f", linewidths=1, linecolor='#0a1120', cbar_kws={"shrink": .8}, vmin=-1, vmax=1, annot_kws={"size": 10, "weight": "bold"}, ax=ax1)

        feature_names = ['Số Môn', 'Giờ Học', 'Đi Làm', 'Giờ Ngủ', 'CLB', 'Đi Học %', 'Giờ MXH', 'GPA', 'Học Nhóm']
        ax1.set_xticklabels(feature_names, rotation=45, ha='right', color='#cbd5e1')
        ax1.set_yticklabels(feature_names, rotation=0, color='#cbd5e1')
        st.pyplot(fig1)

    with col_chart2:
        st.markdown("<h4 style='text-align: center; color: #94a3b8; margin-bottom: 10px;'>Xu Hướng Tuyến Tính Quy Chuẩn</h4>", unsafe_allow_html=True)
        fig2, ax2 = plt.subplots(figsize=(8, 6), facecolor='#0a1120')
        ax2.set_facecolor('#0a1120')
        sns.regplot(x='Gio_Hoc', y='GPA', data=df, ax=ax2, scatter_kws={'alpha':0.5, 'color':'#00f3ff', 's':60}, line_kws={'color':'#ff007f', 'linewidth': 3})
        ax2.set_xlabel('Số Giờ Tự Học / Ngày', color='#94a3b8')
        ax2.set_ylabel('Điểm Số GPA Tích Lũy', color='#94a3b8')
        ax2.tick_params(colors='#cbd5e1')
        ax2.grid(True, linestyle=':', alpha=0.2, color='#94a3b8')
        sns.despine(left=True, bottom=True)
        st.pyplot(fig2)

    st.markdown("---")
    st.markdown("<h4 style='text-align: center; color: #94a3b8; margin-bottom: 15px;'>Bảng Phân Tích Hệ Số Beta Mô Hình Tuyến Tính</h4>", unsafe_allow_html=True)
    coeff_df = pd.DataFrame({'Biến Số Khảo Sát': ['Số Môn Học', 'Giờ Tự Học', 'Đi Làm Thêm', 'Giờ Ngủ', 'Tham Gia CLB', 'Tỷ Lệ Đi Học', 'Giờ Lướt MXH', 'Học Nhóm'], 'Trọng Số Tác Động': model.coef_}).sort_values(by='Trọng Số Tác Động', ascending=False)
    st.dataframe(coeff_df.style.background_gradient(cmap='coolwarm', axis=0).format({"Trọng Số Tác Động": "{:+.4f}"}), use_container_width=True)

In [ ]:
!pip install pyngrok
from pyngrok import ngrok
ngrok.set_auth_token("3DqD71SQqdUgkYlXSsM2B4InGjy_3BkWt1EBmEyt2CYNfypnn")
ngrok.kill()

!nohup streamlit run app.py --server.port 8501 &
import time
time.sleep(4)
public_url = ngrok.connect(8501)
print(f"\n🚀 SYSTEM ONLINE! TRUY CẬP LINK WEB SANG TRỌNG CỦA BẠN TẠI ĐÂY:")
print(f"👉 {public_url}")